# 05 — PPDA (Passes Per Defensive Action)

**Objective:** Measure team pressing intensity across a Serie A season.

**Definition:**
$$\text{PPDA} = \frac{\text{Opponent passes in their own defensive zone}}{\text{Pressing team's ball recoveries in that zone}}$$

- **Lower PPDA → more intense pressing** (fewer opponent passes allowed before winning the ball back).
- **Zone:** the opponent's own defensive 60% of the pitch (from their goal to ~60% up).
  Equivalently: events where the *passer's* `x_from_own_goal ≤ 60`.
- **Denominator:** `Ball recovery` — the Opta event fired every time a team gains
  possession of a loose ball (after tackles, interceptions, misplaced passes, etc.).
  This is the cleanest single event that marks "possession won".

**Outputs:**

1. Ranked PPDA table + horizontal bar chart2. Interactive scatter plot: Mean seconds to regain (x) vs PPDA (y) with team badges

In [9]:
# ============================================================
# 0 · CONFIGURATION
# ============================================================
# Edit ONLY this cell to change season, paths, or pressing zone.

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Paths (relative to this notebook's location) ─────────────
REPO_ROOT  = Path.cwd().parent                       # FMP_SerieA_Dashboard/
SEASON     = "2024_2025"                              # ← change season here
DATA_DIR   = REPO_ROOT / "data" / "raw" / f"serie_a_{SEASON}" / "events"
OUTPUT_DIR = REPO_ROOT / "outputs"
LOGOS_DIR  = REPO_ROOT / "docs" / "logos" / "seriea"  # team badge PNGs

assert DATA_DIR.exists(), f"Data folder not found: {DATA_DIR}"

# ── PPDA pressing zone ──────────────────────────────────────
# The zone is defined from the PASSER's perspective (the opponent building up).
# We count opponent passes where passer's x_from_own_goal ≤ PPDA_ZONE_UPPER.
# 60 = opponent's own defensive 60% (standard Understat / StatsBomb definition)
PPDA_ZONE_UPPER = 60

# ── Event definitions ─────────────────────────────────────────
PASS_EVENT    = "Pass"
REGAIN_EVENT  = "Ball recovery"     # Opta event = possession won

# ── Team name → badge filename mapping ───────────────────────
# Includes teams from multiple seasons (2024/25, 2025/26, etc.)
TEAM_BADGE_MAP = {
    # Core teams (typically in all recent seasons)
    "Atalanta Bergamasca Calcio": "atalanta",
    "Bologna FC 1909":            "bologna",
    "ACF Fiorentina":             "fiorentina",
    "Genoa CFC":                  "genoa",
    "FC Internazionale Milano":   "inter",
    "Juventus FC":                "juventus",
    "SS Lazio":                   "lazio",
    "AC Milan":                   "milan",
    "SSC Napoli":                 "napoli",
    "AS Roma":                    "roma",
    "Torino FC":                  "torino",
    "Udinese Calcio":             "udinese",
    # Variable teams (relegated/promoted)
    "Cagliari Calcio":            "cagliari",
    "Calcio Como 1907":           "como",
    "US Cremonese":               "cremonese",
    "Hellas Verona FC":           "hellasverona",
    "US Lecce":                   "lecce",
    "Parma Calcio 1913":          "parma",
    "Pisa Sporting Club":         "pisa",
    "US Sassuolo Calcio":         "sassuolo",
    # Other seasons
    "Venezia FC":                 "venezia",
    "Empoli FC":                  "empoli",
    "AC Monza":                   "monza",
    "US Salernitana 1919":        "salernitana",
    "Frosinone Calcio":           "frosinone",
}

# Short display names for charts
TEAM_SHORT_NAME = {
    # Core teams
    "Atalanta Bergamasca Calcio": "Atalanta",
    "Bologna FC 1909":            "Bologna",
    "ACF Fiorentina":             "Fiorentina",
    "Genoa CFC":                  "Genoa",
    "FC Internazionale Milano":   "Inter",
    "Juventus FC":                "Juventus",
    "SS Lazio":                   "Lazio",
    "AC Milan":                   "Milan",
    "SSC Napoli":                 "Napoli",
    "AS Roma":                    "Roma",
    "Torino FC":                  "Torino",
    "Udinese Calcio":             "Udinese",
    # Variable teams
    "Cagliari Calcio":            "Cagliari",
    "Calcio Como 1907":           "Como",
    "US Cremonese":               "Cremonese",
    "Hellas Verona FC":           "Hellas Verona",
    "US Lecce":                   "Lecce",
    "Parma Calcio 1913":          "Parma",
    "Pisa Sporting Club":         "Pisa",
    "US Sassuolo Calcio":         "Sassuolo",
    # Other seasons
    "Venezia FC":                 "Venezia",
    "Empoli FC":                  "Empoli",
    "AC Monza":                   "Monza",
    "US Salernitana 1919":        "Salernitana",
    "Frosinone Calcio":           "Frosinone",
}

print(f"Season        : {SEASON}")
print(f"Data folder   : {DATA_DIR}")
print(f"PPDA zone     : passer's x_from_own_goal ≤ {PPDA_ZONE_UPPER}")
print(f"Regain event  : {REGAIN_EVENT}")
print(f"Output folder : {OUTPUT_DIR}")

Season        : 2024_2025
Data folder   : /Users/ricki/Local Projects/FMP_SerieA_Dashboard/data/raw/serie_a_2024_2025/events
PPDA zone     : passer's x_from_own_goal ≤ 60
Regain event  : Ball recovery
Output folder : /Users/ricki/Local Projects/FMP_SerieA_Dashboard/outputs


In [10]:
# ============================================================
# 1 · LOAD & CLEAN EVENT DATA
# ============================================================

import pandas as pd
import numpy as np

# ── Load all match CSVs ──────────────────────────────────────
COLS = [
    "event_id", "event", "period_id", "time_min", "time_sec",
    "team_name", "team_position", "x", "y", "outcome", "match_id",
]
files = sorted(DATA_DIR.glob("*.csv"))
print(f"CSV files found: {len(files)}")

events = pd.concat(
    [pd.read_csv(f, usecols=COLS) for f in files],
    ignore_index=True,
)

# ── Numeric coercions ────────────────────────────────────────
for col in ["period_id", "time_min", "time_sec", "x", "y"]:
    events[col] = pd.to_numeric(events[col], errors="coerce")

# ── Drop rows missing essentials ─────────────────────────────
events = events.dropna(
    subset=["match_id", "team_name", "team_position", "period_id", "time_min", "time_sec", "x"]
)

# ── Outcome → boolean ───────────────────────────────────────
events["is_success"] = events["outcome"].apply(
    lambda v: int(v) == 1 if pd.notna(v) and isinstance(v, (int, float, np.integer, np.floating)) else False
)

# ── Timestamp in seconds (within each half) ──────────────────
events["t_sec"] = events["time_min"] * 60 + events["time_sec"]

# ── Event flags ──────────────────────────────────────────────
events["is_pass"]   = events["event"].str.strip().str.lower().eq(PASS_EVENT.lower())
events["is_regain"] = events["event"].str.strip().str.lower().eq(REGAIN_EVENT.lower())

print(f"Total events loaded: {len(events):,}")
print(f"Matches: {events['match_id'].nunique()}")
print(f"Teams:   {events['team_name'].nunique()}")
events[["event", "is_pass", "is_regain"]].drop_duplicates().sort_values("event").head(20)

CSV files found: 380
Total events loaded: 626,130
Matches: 380
Teams:   20


,event,is_pass,is_regain
115,Aerial,False,False
23,Ball recovery,False,True
13,Ball touch,False,False
134,Blocked Pass,False,False
1166,Card,False,False
155,Challenge,False,False
8656,Chance missed,False,False
2887,Claim,False,False
153,Clearance,False,False
79696,Coach Setup,False,False


In [11]:
# ============================================================
# 2 · OPPONENT MAPPING & x_from_own_goal
# ============================================================

# ── Build home/away lookup per match ─────────────────────────
teams_by_match = (
    events[["match_id", "team_name", "team_position"]]
    .drop_duplicates()
    .pivot_table(index="match_id", columns="team_position", values="team_name", aggfunc="first")
)
teams_by_match.columns = [str(c).strip().lower() for c in teams_by_match.columns]

assert {"home", "away"}.issubset(teams_by_match.columns), \
    f"Expected 'home'/'away' positions, got: {list(teams_by_match.columns)}"

match_home = teams_by_match["home"].to_dict()
match_away = teams_by_match["away"].to_dict()

def _get_opponent(match_id, team_name):
    h, a = match_home.get(match_id), match_away.get(match_id)
    if team_name == h: return a
    if team_name == a: return h
    return np.nan

events["opponent"] = [
    _get_opponent(mid, tn)
    for mid, tn in zip(events["match_id"], events["team_name"])
]
events = events.dropna(subset=["opponent"])

# ── Compute x_from_own_goal ─────────────────────────────────
# Convention:
#   Period 1 → Home attacks right (own goal x=0), Away attacks left (own goal x=100)
#   Period 2+ → directions swap
def _own_goal_x(team_position: str, period_id: int) -> int:
    tp = str(team_position).strip().lower()
    p = int(period_id)
    if p == 1:
        return 0 if tp == "home" else 100
    else:
        return 100 if tp == "home" else 0

events["own_goal_x"] = [
    _own_goal_x(tp, p)
    for tp, p in zip(events["team_position"], events["period_id"])
]
events["x_from_own_goal"] = np.where(
    events["own_goal_x"] == 0, events["x"], 100 - events["x"]
)

# ── Add short display names ──────────────────────────────────
events["team_short"] = events["team_name"].map(TEAM_SHORT_NAME).fillna(events["team_name"])

print(f"Events with opponent & x_from_own_goal: {len(events):,}")

Events with opponent & x_from_own_goal: 626,130


In [12]:
# ============================================================
# 3 · COMPUTE PPDA (SEASON-LEVEL, PER TEAM)
# ============================================================
#
# PPDA for Team A = (opponent passes in opponent's own defensive zone)
#                   / (Team A's ball recoveries in that same zone)
#
# ZONE (from the PASSER's perspective):
#   "opponent's own defensive 60%" → passer's x_from_own_goal ≤ 60
#
# From the PRESSING team's perspective, the same physical area is
#   presser's x_from_own_goal ≥ (100 - 60) = 40.
#
# DENOMINATOR — "Ball recovery" (Opta):
#   Fired every time a team gains possession of a loose ball, whether
#   after a won tackle, interception, misplaced opponent pass, etc.
#   It is the single cleanest marker for "possession won".

# ── 1) Opponent passes in opponent's own defensive zone ──────
passes_in_zone = events[
    events["is_pass"]
    & (events["x_from_own_goal"] <= PPDA_ZONE_UPPER)
].copy()

passes_in_zone["pressing_team"] = passes_in_zone["opponent"]

passes_allowed = (
    passes_in_zone.groupby("pressing_team")
    .size()
    .rename("passes_allowed")
)

# ── 2) Pressing team's ball recoveries in the same zone ──────
PRESSING_ZONE_MIN = 100 - PPDA_ZONE_UPPER

regains_in_zone = events[
    events["is_regain"]
    & (events["x_from_own_goal"] >= PRESSING_ZONE_MIN)
]

ball_recoveries = (
    regains_in_zone.groupby("team_name")
    .size()
    .rename("ball_recoveries")
)

# ── 3) Combine → PPDA ───────────────────────────────────────
ppda = pd.concat([passes_allowed, ball_recoveries], axis=1).fillna(0)
ppda = ppda[ppda["ball_recoveries"] > 0].copy()
ppda["PPDA"] = (ppda["passes_allowed"] / ppda["ball_recoveries"]).round(2)

# ── 4) Per-match PPDA (for std dev) ──────────────────────────
pa_match = (
    passes_in_zone.groupby(["pressing_team", "match_id"])
    .size()
    .rename("passes_allowed")
    .reset_index()
)
br_match = (
    regains_in_zone.groupby(["team_name", "match_id"])
    .size()
    .rename("ball_recoveries")
    .reset_index()
    .rename(columns={"team_name": "pressing_team"})
)
ppda_match = pa_match.merge(br_match, on=["pressing_team", "match_id"], how="outer").fillna(0)
ppda_match["ppda_match"] = np.where(
    ppda_match["ball_recoveries"] > 0,
    ppda_match["passes_allowed"] / ppda_match["ball_recoveries"],
    np.nan,
)

ppda_stats = (
    ppda_match.groupby("pressing_team")["ppda_match"]
    .agg(matches="count", ppda_std="std")
    .round(2)
)
ppda = ppda.join(ppda_stats)

# ── Sort & label ─────────────────────────────────────────────
ppda = ppda.sort_values("PPDA", ascending=True)
ppda.index.name = "team"
ppda["team_short"] = ppda.index.map(TEAM_SHORT_NAME)

print(f"PPDA computed for {len(ppda)} teams")
print(f"  Zone: passer's x_from_own_goal ≤ {PPDA_ZONE_UPPER} | presser's ≥ {PRESSING_ZONE_MIN}")
print(f"  Denominator: {REGAIN_EVENT}\n")
ppda

PPDA computed for 20 teams
  Zone: passer's x_from_own_goal ≤ 60 | presser's ≥ 40
  Denominator: Ball recovery



,passes_allowed,ball_recoveries,PPDA,matches,ppda_std,team_short
team,,,,,,
Bologna FC 1909,8657,964,8.98,38,3.27,Bologna
Atalanta Bergamasca Calcio,10596,1084,9.77,38,2.95,Atalanta
Calcio Como 1907,9939,952,10.44,38,4.22,Como
FC Internazionale Milano,9700,921,10.53,38,4.57,Inter
Juventus FC,10167,910,11.17,38,3.81,Juventus
SS Lazio,10402,882,11.79,38,7.91,Lazio
AC Milan,10123,851,11.90,38,4.50,Milan
SSC Napoli,10836,896,12.09,38,5.20,Napoli
AS Roma,10934,887,12.33,38,5.56,Roma


In [13]:
# ============================================================
# 4 · OUTPUT A — RANKED PPDA TABLE + BAR CHART
# ============================================================

import plotly.express as px
import plotly.graph_objects as go

# ── Prepare display DataFrame ────────────────────────────────
ppda_display = ppda.reset_index().rename(columns={
    "team_short": "Team",
    "passes_allowed": "Passes Allowed",
    "ball_recoveries": "Ball Recoveries",
    "PPDA": "PPDA",
    "matches": "Matches",
    "ppda_std": "Std Dev",
})
ppda_display["Rank"] = range(1, len(ppda_display) + 1)
ppda_display = ppda_display[["Rank", "Team", "PPDA", "Passes Allowed", "Ball Recoveries", "Matches", "Std Dev"]]

print(f"Serie A {SEASON.replace('_','/')} — PPDA Ranking")
print(f"Zone: opponent's own defensive {PPDA_ZONE_UPPER}%")
print("Lower PPDA = more intense pressing\n")
display(ppda_display)

# ── Horizontal bar chart ─────────────────────────────────────
fig_bar = px.bar(
    ppda_display.sort_values("PPDA", ascending=True),
    x="PPDA",
    y="Team",
    orientation="h",
    text="PPDA",
    color="PPDA",
    color_continuous_scale="RdYlGn",     # green=low (good pressing), red=high
    labels={"PPDA": "PPDA (passes allowed per ball recovery)"},
    title=f"Serie A {SEASON.replace('_','/')} — PPDA Ranking<br>"
          f"<sup>Zone: opponent's own defensive {PPDA_ZONE_UPPER}% · "
          f"Denominator: {REGAIN_EVENT}</sup>",
)
fig_bar.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig_bar.update_layout(
    yaxis=dict(autorange="reversed", title=""),
    xaxis_title="PPDA",
    template="plotly_white",
    height=max(500, len(ppda_display) * 35),
    coloraxis_showscale=False,
    margin=dict(l=140),
)
fig_bar.show()

Serie A 2024/2025 — PPDA Ranking
Zone: opponent's own defensive 60%
Lower PPDA = more intense pressing



,Rank,Team,PPDA,Passes Allowed,Ball Recoveries,Matches,Std Dev
0,1,Bologna,8.98,8657,964,38,3.27
1,2,Atalanta,9.77,10596,1084,38,2.95
2,3,Como,10.44,9939,952,38,4.22
3,4,Inter,10.53,9700,921,38,4.57
4,5,Juventus,11.17,10167,910,38,3.81
5,6,Lazio,11.79,10402,882,38,7.91
6,7,Milan,11.90,10123,851,38,4.50
7,8,Napoli,12.09,10836,896,38,5.20
8,9,Roma,12.33,10934,887,38,5.56
9,10,Fiorentina,12.53,11023,880,38,4.37


## Pressing Speed: Mean Seconds to Regain

To make the scatter plot meaningful, we add a second pressing dimension beyond PPDA.

**Mean seconds to regain** measures how quickly a team wins the ball back after losing possession — computed inside the same physical zone used for PPDA (pressing team's `x_from_own_goal ≥ 40`).

**Method:**
1. Sort all in-zone events chronologically within each match.
2. Identify each **turnover moment** — when possession switches away from Team A.
3. Identify the **regain moment** — when Team A next records a Ball recovery in the zone.
4. Compute the elapsed seconds between turnover and regain.
5. Average across all such windows per team.

In [14]:
# ============================================================
# 5 · COMPUTE MEAN SECONDS TO REGAIN (PER TEAM)
# ============================================================
# Uses events in the same PPDA zone (pressing team's x_from_own_goal >= 40).
# Detects possession changes and measures time until the pressing team
# records a Ball recovery to win the ball back.

PRESSING_ZONE_MIN = 100 - PPDA_ZONE_UPPER  # same zone as PPDA

events_zone = events[events["x_from_own_goal"] >= PRESSING_ZONE_MIN].copy()

ez = events_zone.sort_values(
    ["match_id", "period_id", "t_sec", "event_id"], kind="mergesort"
).reset_index(drop=True)

# ── Possession proxy ─────────────────────────────────────────
ez["pos_team"] = np.nan
mask_control = (ez["is_pass"] & ez["is_success"]) | ez["is_regain"]
ez.loc[mask_control, "pos_team"] = ez.loc[mask_control, "team_name"]
ez["pos_team"] = ez.groupby("match_id")["pos_team"].ffill()

# ── Detect possession changes ────────────────────────────────
ez["prev_pos_team"] = ez.groupby("match_id")["pos_team"].shift(1)
ez["pos_change"] = (ez["pos_team"] != ez["prev_pos_team"]) & ez["prev_pos_team"].notna()

# ── Build loss → regain windows ──────────────────────────────
loss_events = ez[ez["pos_change"]].copy()
loss_events["lost_team"]   = loss_events["prev_pos_team"]
loss_events["loss_t"]      = loss_events["t_sec"]
loss_events["loss_period"] = loss_events["period_id"]

da_events = ez[ez["is_regain"]][["match_id", "period_id", "t_sec", "team_name"]].copy()
da_events = da_events.rename(columns={"t_sec": "regain_t", "team_name": "regain_team"})

windows = []
for _, loss in loss_events.iterrows():
    mid, team, t0, pid = loss["match_id"], loss["lost_team"], loss["loss_t"], loss["loss_period"]
    candidates = da_events[
        (da_events["match_id"] == mid)
        & (da_events["regain_team"] == team)
        & (da_events["period_id"] == pid)
        & (da_events["regain_t"] > t0)
    ]
    if candidates.empty:
        continue
    windows.append({
        "match_id": mid,
        "team": team,
        "loss_t": t0,
        "regain_t": candidates["regain_t"].iloc[0],
        "seconds_to_regain": candidates["regain_t"].iloc[0] - t0,
    })

regain_df = pd.DataFrame(windows)
print(f"Loss → regain windows found: {len(regain_df):,}")

# ── Team-level aggregation ───────────────────────────────────
regain_team = (
    regain_df.groupby("team")["seconds_to_regain"]
    .agg(n_regains="count", mean_seconds="mean", median_seconds="median")
    .round(2)
    .sort_values("mean_seconds")
)

print("\nTop 10 fastest pressing teams (mean seconds to regain):")
display(regain_team.head(10))

Loss → regain windows found: 29,127

Top 10 fastest pressing teams (mean seconds to regain):


,n_regains,mean_seconds,median_seconds
team,,,
Atalanta Bergamasca Calcio,1615,167.84,78.0
FC Internazionale Milano,1381,181.38,95.0
Empoli FC,1508,183.98,105.0
Bologna FC 1909,1466,192.29,104.0
AS Roma,1471,196.35,104.0
SSC Napoli,1421,200.66,106.0
Genoa CFC,1466,202.88,102.0
Juventus FC,1374,203.63,100.0
Hellas Verona FC,1383,205.59,118.0


In [15]:
# ============================================================
# 6 · OUTPUT B — SCATTER WITH TEAM BADGES
# ============================================================

import base64
from PIL import Image
from io import BytesIO

# ── Merge PPDA + regain data ─────────────────────────────────
scatter_df = (
    ppda[["PPDA", "passes_allowed", "ball_recoveries", "matches", "ppda_std", "team_short"]]
    .reset_index()
    .merge(regain_team.reset_index(), on="team", how="inner")
)

# ── Build figure with invisible markers (badges will replace them) ──
fig = go.Figure()

# Invisible scatter for hover interactivity
fig.add_trace(go.Scatter(
    x=scatter_df["mean_seconds"],
    y=scatter_df["PPDA"],
    mode="markers",
    marker=dict(size=30, opacity=0),  # invisible — badges on top
    hovertext=[
        f"<b>{row['team_short']}</b><br>"
        f"PPDA: {row['PPDA']:.2f}<br>"
        f"Mean sec to regain: {row['mean_seconds']:.1f}<br>"
        f"Median sec: {row['median_seconds']:.1f}<br>"
        f"Passes allowed: {int(row['passes_allowed'])}<br>"
        f"Ball recoveries: {int(row['ball_recoveries'])}<br>"
        f"Regain windows: {int(row['n_regains'])}<br>"
        f"Matches: {int(row['matches'])}"
        for _, row in scatter_df.iterrows()
    ],
    hoverinfo="text",
    showlegend=False,
))

# ── Add team badges as layout images ─────────────────────────
BADGE_SIZE = 0.035  # as fraction of axis range
x_range = scatter_df["mean_seconds"].max() - scatter_df["mean_seconds"].min()
y_range = scatter_df["PPDA"].max() - scatter_df["PPDA"].min()

for _, row in scatter_df.iterrows():
    badge_slug = TEAM_BADGE_MAP.get(row["team"], "")
    badge_path = LOGOS_DIR / f"{badge_slug}.png"

    if not badge_path.exists():
        # Fallback: add text label if badge not found
        fig.add_annotation(
            x=row["mean_seconds"], y=row["PPDA"],
            text=row["team_short"], showarrow=False,
            font=dict(size=8),
        )
        continue

    # Read and encode badge as base64
    img = Image.open(badge_path)
    buf = BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()

    fig.add_layout_image(
        source=f"data:image/png;base64,{b64}",
        x=row["mean_seconds"],
        y=row["PPDA"],
        xref="x", yref="y",
        sizex=x_range * BADGE_SIZE * 2.5,
        sizey=y_range * BADGE_SIZE * 2.5,
        xanchor="center", yanchor="middle",
        layer="above",
    )

# ── Quadrant lines at the median ─────────────────────────────
fig.add_hline(y=scatter_df["PPDA"].median(), line_dash="dash", line_color="grey", opacity=0.4)
fig.add_vline(x=scatter_df["mean_seconds"].median(), line_dash="dash", line_color="grey", opacity=0.4)

# Quadrant annotations
fig.add_annotation(
    x=scatter_df["mean_seconds"].min(), y=scatter_df["PPDA"].min(),
    text="🟢 High press<br>(fast + intense)", showarrow=False,
    font=dict(size=10, color="green"), xanchor="left", yanchor="bottom",
)
fig.add_annotation(
    x=scatter_df["mean_seconds"].max(), y=scatter_df["PPDA"].max(),
    text="🔴 Low press<br>(slow + passive)", showarrow=False,
    font=dict(size=10, color="red"), xanchor="right", yanchor="top",
)

fig.update_layout(
    title=f"Serie A {SEASON.replace('_','/')} — Pressing Intensity<br>"
          f"<sup>PPDA vs Mean Seconds to Regain · "
          f"Zone: opponent's defensive {PPDA_ZONE_UPPER}%</sup>",
    xaxis_title="Mean Seconds to Regain (lower = faster pressing)",
    yaxis_title="PPDA (lower = more intense pressing)",
    template="plotly_white",
    height=700,
    width=950,
    xaxis=dict(range=[
        scatter_df["mean_seconds"].min() - x_range * 0.08,
        scatter_df["mean_seconds"].max() + x_range * 0.08,
    ]),
    yaxis=dict(range=[
        scatter_df["PPDA"].min() - y_range * 0.08,
        scatter_df["PPDA"].max() + y_range * 0.08,
    ]),
)
fig.show()

In [16]:
# # ============================================================
# # 7 · EXPORT RESULTS
# # ============================================================

# import json

# # ── Build export payload ─────────────────────────────────────
# export = {
#     "season": SEASON,
#     "pressing_zone_x_min": PRESSING_ZONE_X_MIN,
#     "def_action_events": sorted(DEF_ACTION_EVENTS),
#     "teams": scatter_df.to_dict(orient="records"),
# }

# # ── Save to outputs/ ─────────────────────────────────────────
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# out_path = OUTPUT_DIR / f"ppda_{SEASON}.json"
# with open(out_path, "w") as f:
#     json.dump(export, f, indent=2, default=str)

# print(f"✅ PPDA results exported to: {out_path}")